#12 レポート用資料

12.1でグループIDと自分の学籍番号の末尾3桁、自分の任意のPINを入力してください。
また、背景事象のカウント数と測定時間も入力してください。青と緑のボタンを押すと、グループIDに紐づいた実験データを取得します。このデータをもとにday1,2の結果を再現します。8.1のモデルは個人IDに紐づき、モデルに関連するプロットはすべてこれを基準に生成されます。レポートに用いる図は原則としてこのノートブックで作られたものを使用して下さい。

In [ ]:
# @title 12.1 個人アカウントと班データの読み込み
import ast
import inspect
import textwrap
import base64
import copy
import io
import json
import linecache
import math
import traceback
from types import SimpleNamespace
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import requests
import scipy.stats as stats
from scipy.optimize import curve_fit
import ipywidgets as widgets
from IPython.display import display, clear_output
GAS_URL = 'https://script.google.com/macros/s/AKfycbzqAFZmMLfyITKDOtJSLw36D-ZvdO9NHIHIX19vsGOQxVJlB6CzRMhobYFTjFPKpmGi_g/exec'
PHYSICS_CONSTANTS = {'N_A': 6.022e+23, 'z_alpha': 2, 'E_alpha_MeV': 4.5, 'e2_MeV_fm': 197.3 / 137.036, 'Z_Au': 79, 'A_Au': 197.0, 'rho_Au': 19.3, 'Z_Al': 13, 'A_Al': 26.98, 'rho_Al': 2.69}
BACKGROUND_COUNTS = 1
BACKGROUND_TIME_SEC = 4515.0
experiment_state = {}
server_data_cache = {'fetched': False, 'all_files': [], 'all_models': []}
group_calculated_constants = {}
active_model_record = None
active_model_code = ''
active_model_kind = ''
feedback_account_id = None
FEEDBACK_PREFIX = '# feedback_requested: '
linked = False

def api_call(action, **fields):
    payload = {'action': action, 'group_id': experiment_state['personal_id'], 'pin': experiment_state['pin']}
    payload.update(fields)
    if payload['group_id'] != experiment_state['personal_id']:
        raise ValueError('All requests must authenticate as the personal ID.')
    response = requests.post(GAS_URL, json=payload, timeout=120)
    response.raise_for_status()
    result = response.json()
    if result.get('status') != 'success':
        raise RuntimeError(result.get('message', 'Server request failed.'))
    return result

def require_link():
    if not linked:
        raise RuntimeError('Enter Group ID, Student ID (last 3 digits), and Personal PIN in 0.1 first.')

def model_caption():
    record_id = active_model_record.get('group_id') if active_model_record else experiment_state.get('group_id', '')
    if active_model_kind == 'Draft':
        record_id = experiment_state.get('personal_id', '')
    return f'Model: {active_model_kind} | ID: {record_id}'

def _to_float(val, default=0.0):
    try:
        if val is None or str(val).strip() == '':
            return default
        return float(val)
    except (ValueError, TypeError):
        return default

def recalculate_state(files=None, target_state=None):
    import binascii
    import warnings
    from scipy.optimize import OptimizeWarning
    state = experiment_state if target_state is None else target_state
    if files is None:
        files = state.get('uploaded_files', {})
    pulser_files = [f for f in files.values() if f.get('type') == 'Pulse Height']
    source_files = [f for f in files.values() if f.get('type') == 'Source']
    thick_files = [f for f in files.values() if f.get('type') == 'Thickness']
    for key in ('cal_slope', 'cal_slope_err', 'cal_offset', 'cal_offset_err', 'B_mes', 'B_mes_err', 'B_0', 'B_0_err', 'd_Au', 'd_Au_err', 'd_Al', 'd_Al_err', 'n_Au', 'n_Au_err', 'n_Al', 'n_Al_err'):
        state[key] = 0.0

    def spectrum(f):
        raw = f.get('raw_data')
        restored_time, restored_count = (None, None)
        if raw is None or len(raw) == 0:
            encoded = f.get('file_content')
            if not encoded:
                raise ValueError('Spectrum is missing (raw_data or file_content required).')
            content = base64.b64decode(encoded, validate=True).decode('utf-8-sig')
            raw, restored_time, restored_count = parse_csv_file(content)
        if len(raw) > 5:
            try:
                float(raw[-1])
            except (ValueError, TypeError):
                raw = raw[:-5]
        y = np.asarray(raw[:1024], dtype=float)
        if y.ndim != 1 or len(y) < 4:
            raise ValueError('Spectrum must contain at least four channels.')
        if not np.all(np.isfinite(y)) or np.any(y < 0) or np.max(y) <= 0:
            raise ValueError('Spectrum counts must be finite, nonnegative and nonzero.')
        return (y, restored_time, restored_count)

    def day1_gaussian(x, a, mu, sigma):
        return a * np.exp(-(x - mu) ** 2 / (2 * sigma ** 2))

    def fit_peak(y, sigma_guess):
        x = np.arange(len(y))
        with warnings.catch_warnings():
            warnings.simplefilter('error', OptimizeWarning)
            popt, pcov = curve_fit(day1_gaussian, x, y, p0=[np.max(y), np.argmax(y), sigma_guess])
        mu, mu_err = (float(popt[1]), float(np.sqrt(pcov[1, 1])))
        if not np.isfinite(mu) or not np.isfinite(mu_err) or (not 0 < mu < len(y)):
            raise ValueError('Gaussian peak center or its uncertainty is invalid.')
        return (mu, mu_err)

    def number(f, key, default=None, positive=False):
        value = f.get(key)
        if value is None or str(value).strip() == '':
            value = default
        if value is None:
            raise ValueError(f'Missing {key}.')
        value = float(value)
        if not np.isfinite(value) or (positive and value <= 0):
            raise ValueError(f'Invalid {key}: {value}.')
        return value
    fit_errors = (ValueError, TypeError, RuntimeError, OverflowError, OptimizeWarning, binascii.Error, UnicodeDecodeError)
    if len(pulser_files) < 2 or not source_files:
        print('WARNING: Day 1 recalculation requires at least two Pulse Height files and one Source file.')
        return
    ch_list, v_list = ([], [])
    for f in pulser_files:
        try:
            y, _, _ = spectrum(f)
            mu, _ = fit_peak(y, 5.0)
            voltage = number(f, 'voltage')
        except fit_errors as e:
            print(f"WARNING: Failed to fit pulser {f.get('file_name', '(unnamed)')}: {e}")
            continue
        ch_list.append(mu)
        v_list.append(voltage)
    if len(ch_list) < 2:
        print('ERROR: Fewer than two pulser spectra were fitted successfully.')
        return
    try:
        order = np.argsort(v_list)
        ch_arr = np.asarray(ch_list)[order]
        v_arr = np.asarray(v_list)[order]
        if np.ptp(ch_arr) <= 0:
            raise ValueError('Pulser peak channels must be distinct.')
        if len(ch_arr) > 2:
            p_v, cov_v = np.polyfit(ch_arr, v_arr, 1, cov=True)
            slope_v_err, offset_v_err = np.sqrt(np.diag(cov_v))
        else:
            p_v = np.polyfit(ch_arr, v_arr, 1)
            slope_v_err, offset_v_err = (np.nan, np.nan)
            print('WARNING: Two pulser points cannot determine calibration errors; errors are NaN, as in Day 1.')
        slope_v, offset_v = p_v
        if not np.all(np.isfinite(p_v)) or slope_v <= 0:
            raise ValueError('Voltage calibration must have a finite, positive slope.')
        src = source_files[0]
        y_src, restored_time, restored_count = spectrum(src)
        src_ch, src_ch_err = fit_peak(y_src, 10.0)
        v_src = slope_v * src_ch + offset_v
        if not np.isfinite(v_src) or v_src <= 0:
            raise ValueError('Source voltage must be positive.')
        v_src_err = np.sqrt((src_ch * slope_v_err) ** 2 + (slope_v * src_ch_err) ** 2 + offset_v_err ** 2)
        k_e_v = 4.5 / v_src
        k_err = k_e_v * v_src_err / v_src
        slope_e, offset_e = (slope_v * k_e_v, offset_v * k_e_v)
        slope_e_err = np.sqrt((k_e_v * slope_v_err) ** 2 + (slope_v * k_err) ** 2)
        offset_e_err = np.sqrt((k_e_v * offset_v_err) ** 2 + (offset_v * k_err) ** 2)
    except fit_errors as e:
        print(f'ERROR: Day 1 calibration failed: {e}')
        return
    state.update({'cal_slope': float(slope_e), 'cal_slope_err': float(slope_e_err), 'cal_offset': float(offset_e), 'cal_offset_err': float(offset_e_err)})
    try:
        t_sec = number(src, 'time_sec', restored_time if restored_time is not None else 1.0, positive=True)
        total_count = number(src, 'total_count', restored_count if restored_count is not None else float(np.sum(y_src)))
        if total_count < 0:
            raise ValueError('Source total_count must be nonnegative.')
        s2 = number(src, 'slit2', 6.0, positive=True)
        dist = number(src, 'distance', 70.0, positive=True)
        solid_angle_ratio = np.pi * (s2 / 2.0) ** 2 / dist ** 2 / (4.0 * np.pi)
        rate, rate_err = (total_count / t_sec, np.sqrt(total_count) / t_sec)
        state.update({'B_mes': float(rate), 'B_mes_err': float(rate_err), 'B_0': float(rate / solid_angle_ratio), 'B_0_err': float(rate_err / solid_angle_ratio)})
    except fit_errors as e:
        print(f'ERROR: Source activity calculation failed: {e}')
    energies = np.array([1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.1, 4.5])
    cheat_ranges = {'Au': [2.699, 3.435, 4.23, 5.086, 6.002, 6.974, 7.175, 8.001], 'Al': [5.017, 6.804, 8.777, 10.944, 13.308, 15.867, 16.402, 18.618]}

    def range_linear(energy, a, b):
        return a * energy + b
    range_fits = {}
    for mat, ranges in cheat_ranges.items():
        try:
            with warnings.catch_warnings():
                warnings.simplefilter('error', OptimizeWarning)
                range_fits[mat] = curve_fit(range_linear, energies, ranges, p0=[1.0, 0.0])
        except fit_errors as e:
            print(f'ERROR: {mat} reference range fit failed: {e}')
    for f in thick_files:
        mat = f.get('material')
        if mat not in range_fits:
            print(f'WARNING: Cannot calculate thickness for material {mat!r}.')
            continue
        try:
            y, _, _ = spectrum(f)
            ch_rem, ch_rem_err = fit_peak(y, 5.0)
            e_rem = slope_e * ch_rem + offset_e
            e_rem_err = np.sqrt((slope_e * ch_rem_err) ** 2 + (ch_rem * slope_e_err) ** 2 + offset_e_err ** 2)
            p_range, cov_range = range_fits[mat]
            a = p_range[0]
            d_um = a * (4.5 - e_rem)
            jacobian = np.array([4.5 - e_rem, 0.0])
            d_err = np.sqrt((a * e_rem_err) ** 2 + jacobian.T @ cov_range @ jacobian)
            density_factor = PHYSICS_CONSTANTS['rho_' + mat] * PHYSICS_CONSTANTS['N_A'] / PHYSICS_CONSTANTS['A_' + mat]
            state.update({'d_' + mat: float(d_um), 'd_' + mat + '_err': float(d_err), 'n_' + mat: float(density_factor * d_um * 0.0001), 'n_' + mat + '_err': float(density_factor * d_err * 0.0001)})
        except fit_errors as e:
            print(f"WARNING: Failed processing thickness {f.get('file_name', '(unnamed)')}: {e}")

def calculate_group_constants(all_files):
    import contextlib
    import io
    global group_calculated_constants
    grouped_files = {}
    excluded_group_ids = {'test', 'ta'}
    current_group_id = str(experiment_state.get('group_id', '')).strip()
    for index, file_data in enumerate(all_files):
        group_id = str(file_data.get('group_id', '')).strip()
        if not group_id or group_id == current_group_id or group_id.lower() in excluded_group_ids:
            continue
        grouped_files.setdefault(group_id, {})[f"{index}:{file_data.get('file_name', 'file')}"] = file_data
    calculated = {}
    required_keys = ('B_0', 'B_0_err', 'n_Au', 'n_Au_err', 'n_Al', 'n_Al_err')
    for group_id, files_by_name in grouped_files.items():
        group_state = {'uploaded_files': files_by_name}
        with contextlib.redirect_stdout(io.StringIO()):
            recalculate_state(files=files_by_name, target_state=group_state)
        b0 = group_state.get('B_0', 0.0)
        densities = [group_state.get('n_Au', 0.0), group_state.get('n_Al', 0.0)]
        if np.isfinite(b0) and b0 > 0 and any((np.isfinite(value) and value > 0 for value in densities)):
            calculated[group_id] = {key: group_state.get(key, 0.0) for key in required_keys}
    group_calculated_constants = calculated
    return (len(calculated), len(grouped_files))

def local_teacher_predict(theta_deg, time_sec, s1, s2, L, isAu):
    E_kin = 4.5
    z_alpha = 2.0
    Z_target = 79.0 if isAu else 13.0
    B_0 = experiment_state.get('B_0', 2690000.0)
    B_0_err = experiment_state.get('B_0_err', 17900.0)
    if isAu:
        n_target = experiment_state.get('n_Au', 5.54e+18)
        n_err = experiment_state.get('n_Au_err', 0.0)
    else:
        n_target = experiment_state.get('n_Al', 7.62e+19)
        n_err = experiment_state.get('n_Al_err', 0.0)
    L_s1 = L - 11.7 if L - 11.7 > 0 else 1.0
    omega_1 = np.pi * (s1 / 2.0) ** 2 / L_s1 ** 2
    omega_2 = np.pi * (s2 / 2.0) ** 2 / 60.0 ** 2
    e2_fm = globals().get('PHYSICS_CONSTANTS', {}).get('e2_MeV_fm', 197.3 / 137.036)
    a_fm = z_alpha * Z_target * e2_fm / (4.0 * E_kin)
    a_cm = a_fm * 1e-13
    theta_rad = np.radians(theta_deg)
    ds_fm2 = a_fm ** 2 / np.sin(theta_rad / 2.0) ** 4 if theta_deg > 0 else 0.0
    incident_rate = B_0 * (omega_1 / (4.0 * np.pi))
    expected_counts = incident_rate * n_target * (ds_fm2 * 1e-26) * omega_2 * time_sec
    stat_err = np.sqrt(expected_counts) if expected_counts > 0 else 0.0
    rel_err_sq = 0.0
    if B_0 and B_0 > 0:
        rel_err_sq += (B_0_err / B_0) ** 2
    if n_target and n_target > 0:
        rel_err_sq += (n_err / n_target) ** 2
    sys_err = expected_counts * np.sqrt(rel_err_sq)
    return (expected_counts, stat_err, sys_err, sys_err)

def calculate_angle_limits(theta_deg, s1, s2, L):
    if not 15.0 <= theta_deg <= 180.0:
        raise ValueError('Angle must be between 15 and 180 degrees.')
    L_s1 = 11.7
    L3 = 60.0
    r_src = 5.0
    dr1 = s1 / 2.0
    dr2 = s2 / 2.0
    theta_rad = np.radians(theta_deg)
    min_angle = 180.0
    max_angle = 0.0
    best_Tp_min, best_Sp_min, best_Dp_min = (None, None, None)
    best_Tp_max, best_Sp_max, best_Dp_max = (None, None, None)
    u_x, u_y = (np.cos(np.pi / 6), np.sin(np.pi / 6))
    for alpha in [-1, 1]:
        for beta in [-1, 1]:
            for gamma in [-1, 1]:
                Sp = np.array([gamma * r_src, -L])
                S1p = np.array([alpha * dr1 * u_x, -L_s1 + alpha * dr1 * u_y])
                V = S1p - Sp
                num = Sp[1] * u_x - Sp[0] * u_y
                den = V[0] * u_y - V[1] * u_x
                if abs(den) > 1e-09:
                    k = num / den
                    Tp = Sp + k * V
                else:
                    Tp = S1p
                vLI = Tp - Sp
                Dc = np.array([-L3 * np.sin(theta_rad), L3 * np.cos(theta_rad)])
                dD = np.array([beta * dr2 * np.cos(theta_rad), beta * dr2 * np.sin(theta_rad)])
                Dp = Dc + dD
                vLO = Dp - Tp
                norm_vLI = np.linalg.norm(vLI)
                norm_vLO = np.linalg.norm(vLO)
                if norm_vLI > 0 and norm_vLO > 0:
                    cos_theta = np.dot(vLI, vLO) / (norm_vLI * norm_vLO)
                    cos_theta = np.clip(cos_theta, -1.0, 1.0)
                    thetap = np.degrees(np.arccos(cos_theta))
                    if thetap < min_angle:
                        min_angle = thetap
                        best_Tp_min, best_Sp_min, best_Dp_min = (Tp, Sp, Dp)
                    if thetap > max_angle:
                        max_angle = thetap
                        best_Tp_max, best_Sp_max, best_Dp_max = (Tp, Sp, Dp)
    return (min_angle, max_angle, best_Tp_min, best_Sp_min, best_Dp_min, best_Tp_max, best_Sp_max, best_Dp_max)

def valid_prediction(prediction):
    return len(prediction) == 4 and np.all(np.isfinite(prediction)) and np.all(np.asarray(prediction) >= 0)

def measurement_count(f):
    raw = f.get('raw_data', [])
    return float(np.sum(raw[:1024])) if len(raw) >= 1024 else float(f.get('total_count', 0.0))

def background_count_estimate(time_sec):
    if 'bg_counts' not in experiment_state or 'bg_time' not in experiment_state:
        raise ValueError('Set background counts and measurement time in 0.1 first.')
    bg_counts = experiment_state['bg_counts']
    bg_time = experiment_state['bg_time']
    if not np.all(np.isfinite([bg_counts, bg_time, time_sec])) or bg_counts < 0 or bg_time <= 0 or (time_sec < 0):
        raise ValueError('Background requires nonnegative counts and positive measurement duration.')
    scale = time_sec / bg_time
    return (scale * bg_counts, scale * np.sqrt(bg_counts))

def observation_with_background(total_count, time_sec):
    bg_expected, bg_error = background_count_estimate(time_sec)
    net = total_count - bg_expected
    error = np.sqrt(total_count + bg_error ** 2)
    return (net, error, error)

def observation_with_dead_time(total_count, time_sec):
    tau = 0.001
    if not np.all(np.isfinite([total_count, time_sec])) or total_count < 0 or time_sec <= 0:
        raise ValueError('Counts must be nonnegative and measurement time must be positive.')
    live_fraction = 1.0 - total_count * tau / time_sec
    if live_fraction <= 0:
        raise ValueError('Dead-time correction requires total_count * tau < time_sec.')
    corrected_count = total_count / live_fraction
    corrected_error = np.sqrt(total_count) / live_fraction ** 2
    bg_expected, bg_error = background_count_estimate(time_sec)
    net = corrected_count - bg_expected
    error = np.sqrt(corrected_error ** 2 + bg_error ** 2)
    return (net, error, error)

def compare_predictions(measured, error_up, error_down, user, teacher):
    p, p_stat, p_sys_up, p_sys_down = user
    t, t_stat, t_sys_up, t_sys_down = teacher
    sigma_user = np.sqrt(p_stat ** 2 + p_sys_up ** 2 + error_down ** 2) if measured >= p else np.sqrt(p_stat ** 2 + p_sys_down ** 2 + error_up ** 2)
    sigma_teacher = np.sqrt(t_stat ** 2 + t_sys_up ** 2 + error_down ** 2) if measured >= t else np.sqrt(t_stat ** 2 + t_sys_down ** 2 + error_up ** 2)
    if not np.all(np.isfinite([sigma_user, sigma_teacher])) or sigma_user <= 0 or sigma_teacher <= 0:
        raise ValueError('Comparison requires finite, positive uncertainties.')
    pull = (measured - p) / sigma_user
    chi_user = pull ** 2
    chi_teacher = ((measured - t) / sigma_teacher) ** 2
    log_user = np.log(2.0 * np.pi * sigma_user ** 2)
    log_teacher = np.log(2.0 * np.pi * sigma_teacher ** 2)
    return (pull, -0.5 * (chi_user - chi_teacher + log_user - log_teacher), chi_user, log_user, chi_teacher, log_teacher)
DEFAULT_MODEL_CODE = 'def predict_counts_with_error(theta_deg, time, s1, s2, L, isAu):\n    """Return (expected counts, statistical error, systematic + error, systematic - error)."""\n    if not 15.0 <= theta_deg <= 180.0:\n        raise ValueError(\'Angle must be between 15 and 180 degrees.\')\n    e2_fm = 197.3 / 137.036\n    z_alpha = 2.0\n    Z_target = 79.0 if isAu else 13.0\n    theta_rad = np.deg2rad(theta_deg)\n    E_kin = 4.5\n    B_0 = experiment_state.get(\'B_0\', 0.0)\n    B_0_err = experiment_state.get(\'B_0_err\', 0.0)\n    if isAu:\n        n_target = experiment_state.get(\'n_Au\', 0.0)\n        n_err = experiment_state.get(\'n_Au_err\', 0.0)\n    else:\n        n_target = experiment_state.get(\'n_Al\', 0.0)\n        n_err = experiment_state.get(\'n_Al_err\', 0.0)\n    n_target *= 1e-26\n    n_err *= 1e-26\n    omega_1 = np.pi * (s1 / 2 / (L - 11.7)) ** 2\n    omega_2 = np.pi * (s2 / 2 / 60) ** 2\n    if isAu:\n        a_fm = z_alpha * Z_target * e2_fm / (4.0 * E_kin)\n    else:\n        a_fm = z_alpha * Z_target * e2_fm / (4.0 * E_kin)\n    t_min, t_max, _, _, _, _, _, _ = calculate_angle_limits(theta_deg, s1, s2, L)\n    theta_rad = np.radians(theta_deg)\n    t_rad_min = np.radians(max(0.1, t_min))\n    t_rad_max = np.radians(min(179.9, t_max))\n    dsigma_domega = a_fm ** 2 / np.sin(theta_rad / 2.0) ** 4\n    ds_min = a_fm ** 2 / np.sin(t_rad_max / 2.0) ** 4\n    ds_max = a_fm ** 2 / np.sin(t_rad_min / 2.0) ** 4\n    incident_rate = B_0 * (omega_1 / (4.0 * np.pi))\n    expected_counts = incident_rate * n_target * dsigma_domega * omega_2 * time\n    geometry_err_plus = incident_rate * n_target * abs(ds_max - dsigma_domega) * omega_2 * time\n    geometry_err_minus = incident_rate * n_target * abs(ds_min - dsigma_domega) * omega_2 * time\n    count_factor = omega_1 / (4.0 * np.pi) * dsigma_domega * omega_2 * time\n    propagated_variance = (count_factor * n_target * B_0_err) ** 2 + (count_factor * B_0 * n_err) ** 2\n    sys_err_plus = np.sqrt(propagated_variance + geometry_err_plus ** 2)\n    sys_err_minus = np.sqrt(propagated_variance + geometry_err_minus ** 2)\n    stat_err = np.sqrt(expected_counts) if expected_counts > 0 else 0.0\n    return (expected_counts, stat_err, sys_err_plus, sys_err_minus)'

def compile_model(source):
    tree = ast.parse(source)
    for node in tree.body:
        if isinstance(node, ast.FunctionDef):
            if node.decorator_list or node.args.defaults or any((v is not None for v in node.args.kw_defaults)):
                raise ValueError('Use plain function definitions without decorators or default expressions.')
        elif isinstance(node, (ast.Import, ast.ImportFrom)):
            modules = [a.name for a in node.names] if isinstance(node, ast.Import) else [node.module or '']
            if any((m.split('.')[0] not in {'numpy', 'scipy', 'math'} for m in modules)):
                raise ValueError('Model imports must use numpy, scipy, or math.')
        elif isinstance(node, ast.Assign):
            ast.literal_eval(node.value)
            if any((not isinstance(t, ast.Name) for t in node.targets)):
                raise ValueError('Top-level assignments must define constants.')
        elif isinstance(node, ast.Expr) and isinstance(node.value, ast.Constant) and isinstance(node.value.value, str):
            pass
        else:
            raise ValueError('Model code must contain functions, numeric constants, and scientific imports only.')
    namespace = {'np': np, 'math': math, 'stats': stats, 'curve_fit': curve_fit, 'experiment_state': copy.deepcopy(experiment_state), 'PHYSICS_CONSTANTS': dict(PHYSICS_CONSTANTS), 'calculate_angle_limits': calculate_angle_limits}
    filename = '<day4-personal-model>'
    linecache.cache[filename] = (len(source), None, source.splitlines(True), filename)
    exec(compile(tree, filename, 'exec'), namespace)
    func = namespace.get('predict_counts_with_error')
    if not callable(func):
        raise ValueError('Define predict_counts_with_error(theta_deg, time, s1, s2, L, isAu).')
    for is_au in (True, False):
        for theta in (20.0, 60.0, 120.0):
            if not valid_prediction(func(theta, 600.0, 6.0, 6.0, 70.0, is_au)):
                raise ValueError('The model must return four finite, nonnegative values.')
    return func

def split_feedback_metadata(source):
    requested = None
    code_lines = []
    for line in source.splitlines(keepends=True):
        if line.startswith(FEEDBACK_PREFIX) and line[len(FEEDBACK_PREFIX):].strip() in ('true', 'false'):
            requested = line[len(FEEDBACK_PREFIX):].strip() == 'true'
        else:
            code_lines.append(line)
    return ''.join(code_lines), requested

def model_code_with_feedback(source, requested):
    code, _ = split_feedback_metadata(source)
    return FEEDBACK_PREFIX + json.dumps(bool(requested)) + '\n' + code

def choose_model(all_models):
    by_id = {str(m.get('group_id', '')): m for m in all_models}
    personal = by_id.get(experiment_state['personal_id'])
    group = by_id.get(experiment_state['group_id'])
    if personal is not None:
        record, kind = (personal, 'Personal')
    elif group is not None:
        record, kind = (group, 'Group')
    else:
        record, kind = (None, 'Group baseline (Day2 default)')
    source = record.get('model_code', '') if record is not None else DEFAULT_MODEL_CODE
    source, _ = split_feedback_metadata(source)
    if not source.strip():
        raise ValueError(f'{kind} model code is empty. Repair the saved model before continuing.')
    return (record, kind, source, compile_model(source))


def activate_model_definition(func):
    global active_model_code, active_model_kind, active_model_record, predict_counts_with_error
    try:
        source = textwrap.dedent(inspect.getsource(func))
    except (OSError, TypeError):
        shell = get_ipython()
        tree = ast.parse(shell.history_manager.input_hist_raw[-1])
        definition = next(node for node in tree.body if isinstance(node, ast.FunctionDef) and node.name == 'predict_counts_with_error')
        source = ast.unparse(definition)
    source = ast.unparse(ast.parse(source))
    if not linked:
        print('Link your account in 0.1, then run 8.1 to apply any edits.')
        return
    if source == ast.unparse(ast.parse(DEFAULT_MODEL_CODE)):
        predict_counts_with_error = compile_model(active_model_code)
        print(model_caption())
        print('The unchanged example keeps the loaded model. Edit the function in 8.1 to replace it.')
        return
    try:
        candidate = compile_model(source)
    except Exception:
        predict_counts_with_error = compile_model(active_model_code)
        raise
    predict_counts_with_error = candidate
    active_model_code = source
    active_model_kind = 'Draft'
    active_model_record = None
    print(model_caption())
    print('Model edits applied locally. Run 8.2 to evaluate and upload your personal model.')

def refresh_views():
    for name in ('run_calibration', 'run_thickness', 'render_trajectory', 'render_geometry', 'run_model_evaluation', 'plot_raw_spectra'):
        func = globals().get(name)
        if callable(func):
            try:
                func(None)
            except Exception as exc:
                print(f'[ERROR] {name}: {exc}')
    if 'run_estimation' in globals():
        run_estimation(None)
input_group = widgets.Text(description='Group ID:', style={'description_width': 'initial'})
input_student = widgets.Text(description='Student ID (last 3 digits):', style={'description_width': 'initial'}, layout=widgets.Layout(width='380px'))
input_pin = widgets.Password(description='Personal PIN:', style={'description_width': 'initial'})
checkbox_feedback = widgets.Checkbox(value=False, description='Request feedback', indent=False)
input_bg_counts = widgets.IntText(value=BACKGROUND_COUNTS, description='Background counts:', style={'description_width': 'initial'})
input_bg_time = widgets.FloatText(value=BACKGROUND_TIME_SEC, description='Background time (s):', style={'description_width': 'initial'})
btn_apply_background = widgets.Button(description='Apply Background', button_style='info')
out_background = widgets.Output()

def background_input_values():
    counts, duration = input_bg_counts.value, input_bg_time.value
    if counts < 0 or not np.isfinite(duration) or duration <= 0:
        raise ValueError('Background counts must be nonnegative and measurement time must be finite and greater than zero.')
    return {'bg_counts': counts, 'bg_time': duration}

def apply_background(_):
    with out_background:
        clear_output(wait=True)
        try:
            values = background_input_values()
            experiment_state.update(values)
            model = globals().get('predict_counts_with_error')
            if callable(model):
                model.__globals__.get('experiment_state', {}).update(values)
            print(f"Background: {values['bg_counts']} count / {values['bg_time']:g} s.")
            print('Applied locally. Run 8.2 to update and upload your metrics, or 9.1 to refresh predictions.')
        except ValueError as exc:
            print(f'[ERROR] {exc}')

btn_apply_background.on_click(apply_background)

btn_link = widgets.Button(description='Link & Generate Figures', button_style='success', layout=widgets.Layout(width='240px'))
out_auth = widgets.Output()

def handle_link_account(_):
    global experiment_state, linked, server_data_cache, feedback_account_id
    global active_model_record, active_model_code, active_model_kind, predict_counts_with_error
    with out_auth:
        clear_output(wait=True)
        group_id, student_id, pin = (input_group.value.strip(), input_student.value.strip(), input_pin.value)
        if not group_id or not pin or len(student_id) != 3 or (not student_id.isascii()) or (not student_id.isdecimal()):
            print('Enter Group ID, exactly three student ID digits, and a nonempty Personal PIN.')
            return
        try:
            background_values = background_input_values()
        except ValueError as exc:
            print(f'[ERROR] {exc}')
            return
        linked = False
        experiment_state = {'group_id': group_id, 'student_id': student_id, 'personal_id': f'{group_id}_{student_id}', 'pin': pin, 'gas_url': GAS_URL, 'uploaded_files': {}, **background_values}
        try:
            api_call('register_group')
            result = api_call('get_all_groups_data')
            all_files, all_models = (result.get('all_files', []), result.get('all_models', []))
            own_files = {f"{i}:{f.get('file_name', '')}": dict(f) for i, f in enumerate(all_files) if str(f.get('group_id', '')) == group_id}
            if not own_files:
                raise ValueError(f'No measurements exist for Group ID {group_id}. Check the spelling and case.')
            if any((str(f.get('group_id', '')) == experiment_state['personal_id'] for f in all_files)):
                raise ValueError('Personal ID conflicts with an existing measurement group.')
            experiment_state['uploaded_files'] = own_files
            recalculate_state()
            for key in ('cal_slope', 'B_0', 'n_Au', 'n_Al'):
                if not np.isfinite(experiment_state.get(key, 0)) or experiment_state.get(key, 0) <= 0:
                    raise ValueError(f'Group calibration data cannot determine {key}.')
            active_model_record, active_model_kind, active_model_code, predict_counts_with_error = choose_model(all_models)
            stored_feedback = None
            if active_model_kind == 'Personal':
                _, stored_feedback = split_feedback_metadata(active_model_record['model_code'])
            if stored_feedback is not None:
                checkbox_feedback.value = stored_feedback
            elif feedback_account_id not in (None, experiment_state['personal_id']):
                checkbox_feedback.value = False
            feedback_account_id = experiment_state['personal_id']
            calculate_group_constants(all_files)
            server_data_cache = {'fetched': True, 'all_files': all_files, 'all_models': all_models}
            linked = True
            input_pin.value = ''
            print(f"Measurements: {group_id} | Personal save ID: {experiment_state['personal_id']}")
            print(model_caption())
            print(f"Background: {experiment_state['bg_counts']} count / {experiment_state['bg_time']:g} s.")
            refresh_views()
        except Exception as exc:
            print(f'[ERROR] {exc}')
btn_link.on_click(handle_link_account)
display(widgets.VBox([input_group, input_student, input_pin, checkbox_feedback, widgets.HBox([input_bg_counts, input_bg_time]), btn_apply_background, out_background, btn_link, out_auth]))


In [ ]:
# @title 1.1 電圧キャリブレーションと線源強度測定
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import ipywidgets as widgets
from IPython.display import display, clear_output
import traceback
import urllib.request
import json

def gaussian(x, a, mu, sigma):
    return a * np.exp(-(x - mu) ** 2 / (2 * sigma ** 2))

def get_clean_data(f_dict):
    raw = f_dict.get('raw_data', [])
    if not raw:
        return np.array([])
    if len(raw) > 5:
        try:
            float(raw[-1])
        except (ValueError, TypeError):
            raw = raw[:-5]
    cleaned = [float(v) for v in raw if str(v).replace('.', '', 1).isdigit()]
    return np.array(cleaned[:1024])
btn_calib = widgets.Button(description='Run Calibration', button_style='primary')
out_calib = widgets.Output()

def run_calibration(b):
    require_link()
    with out_calib:
        clear_output()
        print('--- Calibration Process Started ---')
        try:
            pulse_files = [f for f in experiment_state['uploaded_files'].values() if f['type'] == 'Pulse Height']
            source_files = [f for f in experiment_state['uploaded_files'].values() if f['type'] == 'Source']
            if not pulse_files:
                print("ERROR: No 'Pulse Height' data uploaded for calibration. Aborting.")
                return
            has_source = len(source_files) > 0
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
            results = []
            source_channel_fit, source_err_fit, source_channel_man = (0.0, 0.0, 0.0)
            ax1.set_title('Spectra & Gaussian Fits')
            ax1.set_xlabel('Channel')
            ax1.set_ylabel('Counts')
            if has_source:
                src_data = source_files[0]
                y_src = get_clean_data(src_data)
                x_src = np.arange(len(y_src))
                val_man = src_data.get('channel')
                source_channel_man = float(val_man) if val_man is not None else 0.0
                if len(y_src) > 0:
                    mu_guess, a_guess = (np.argmax(y_src), np.max(y_src))
                    try:
                        popt_src, pcov_src = curve_fit(gaussian, x_src, y_src, p0=[a_guess, mu_guess, 10.0])
                        source_channel_fit = float(popt_src[1])
                        source_err_fit = float(np.sqrt(np.diag(pcov_src))[1])
                        ax1.plot(x_src, y_src, label='Source Data', alpha=0.3, color='black', linestyle='-')
                        ax1.plot(x_src, gaussian(x_src, *popt_src), label=f'Source Fit (mu={source_channel_fit:.1f})', alpha=1.0, color='black', linestyle='--')
                    except Exception as e:
                        print(f'WARNING: Failed to fit Source spectrum. Error: {e}')
                        has_source = False
            colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
            color_idx = 0
            for p in pulse_files:
                y_p = get_clean_data(p)
                if len(y_p) == 0:
                    continue
                x_p, mu_g, a_g = (np.arange(len(y_p)), np.argmax(y_p), np.max(y_p))
                vol = float(p.get('voltage', 0.0))
                ch_man_in = float(p.get('channel', 0.0))
                try:
                    popt, pcov = curve_fit(gaussian, x_p, y_p, p0=[a_g, mu_g, 5.0])
                    mu_fit, err_fit = (float(popt[1]), float(np.sqrt(np.diag(pcov))[1]))
                    results.append({'Voltage (mV)': vol, 'Manual Input (ch)': ch_man_in, 'Fit Center (ch)': mu_fit, 'Fit Error': err_fit})
                    c = colors[color_idx % len(colors)]
                    ax1.plot(x_p, y_p, alpha=0.3, color=c, linestyle='-', label=f'Pulser {vol}mV')
                    ax1.plot(x_p, gaussian(x_p, *popt), alpha=1.0, color=c, linestyle='--', label=f'Fit {vol}mV')
                    color_idx += 1
                except:
                    print(f'WARNING: Failed to fit pulser data for {vol} mV.')
            ax1.legend()
            if not results:
                return
            df_res = pd.DataFrame(results).sort_values('Voltage (mV)')
            v_arr = df_res['Voltage (mV)'].values.astype(float)
            ch_man = df_res['Manual Input (ch)'].values.astype(float)
            ch_fit = df_res['Fit Center (ch)'].values.astype(float)
            err_fit_arr = df_res['Fit Error'].values.astype(float)
            if len(v_arr) > 2:
                p_fit, cov_fit = np.polyfit(ch_fit, v_arr, 1, cov=True)
                slope_fit, offset_fit = (float(p_fit[0]), float(p_fit[1]))
                slope_err_fit, offset_err_fit = (np.sqrt(cov_fit[0, 0]), np.sqrt(cov_fit[1, 1]))
                p_man, cov_man = np.polyfit(ch_man, v_arr, 1, cov=True)
                slope_man, offset_man = (float(p_man[0]), float(p_man[1]))
                slope_err_man, offset_err_man = (np.sqrt(cov_man[0, 0]), np.sqrt(cov_man[1, 1]))
            else:
                p_fit = np.polyfit(ch_fit, v_arr, 1)
                slope_fit, offset_fit = (float(p_fit[0]), float(p_fit[1]))
                p_man = np.polyfit(ch_man, v_arr, 1)
                slope_man, offset_man = (float(p_man[0]), float(p_man[1]))
                slope_err_fit, offset_err_fit = (np.nan, np.nan)
                slope_err_man, offset_err_man = (np.nan, np.nan)
            ax2.set_title('Linear Fit Calibration')
            ax2.set_xlabel('Channel (ch)')
            ax2.set_ylabel('Voltage (mV)', color='C0')
            ax2.errorbar(ch_man, v_arr, xerr=1.0, fmt='o', label='Manual Data', color='lightblue')
            ax2.errorbar(ch_fit, v_arr, xerr=err_fit_arr, fmt='s', label='Gaussian Fit Data', color='C0')
            max_ch = max(np.max(ch_man), np.max(ch_fit)) if len(ch_man) > 0 else 1024
            ch_line = np.linspace(0, max_ch * 1.1, 100)
            ax2.plot(ch_line, slope_man * ch_line + offset_man, color='lightblue', linestyle='--', alpha=0.7, label='Manual Line')
            ax2.plot(ch_line, slope_fit * ch_line + offset_fit, color='C0', linestyle='-', alpha=0.7, label='Fit Line')
            for ch_val, v_val in zip(ch_fit, v_arr):
                ax2.annotate(f'{int(v_val)} mV', (ch_val, v_val), xytext=(5, -12), textcoords='offset points', color='C0', fontsize=9)
            ax2.tick_params(axis='y', labelcolor='C0')
            k_e_v_man = 0.0

            def fmt_err(val, err, dec=4):
                if pd.isna(err) or np.isinf(err):
                    return f'{val:.{dec}f} ± N/A'
                return f'{val:.{dec}f} ± {err:.{dec}f}'
            if has_source and source_channel_fit > 0:
                v_source_man = slope_man * source_channel_man + offset_man
                v_source_err_man = np.sqrt((source_channel_man * slope_err_man) ** 2 + (slope_man * 1.0) ** 2 + offset_err_man ** 2)
                k_e_v_man = 4.5 / v_source_man if v_source_man != 0 else 0.0
                k_err_man = k_e_v_man * (v_source_err_man / v_source_man) if v_source_man != 0 else 0.0
                slope_e_man = slope_man * k_e_v_man
                slope_e_err_man = np.sqrt((k_e_v_man * slope_err_man) ** 2 + (slope_man * k_err_man) ** 2)
                offset_e_man = offset_man * k_e_v_man
                offset_e_err_man = np.sqrt((k_e_v_man * offset_err_man) ** 2 + (offset_man * k_err_man) ** 2)
                v_source_fit = slope_fit * source_channel_fit + offset_fit
                v_source_err_fit = np.sqrt((source_channel_fit * slope_err_fit) ** 2 + (slope_fit * source_err_fit) ** 2 + offset_err_fit ** 2)
                k_e_v_fit = 4.5 / v_source_fit if v_source_fit != 0 else 0.0
                k_err_fit = k_e_v_fit * (v_source_err_fit / v_source_fit) if v_source_fit != 0 else 0.0
                slope_e_fit = slope_fit * k_e_v_fit
                slope_e_err_fit = np.sqrt((k_e_v_fit * slope_err_fit) ** 2 + (slope_fit * k_err_fit) ** 2)
                offset_e_fit = offset_fit * k_e_v_fit
                offset_e_err_fit = np.sqrt((k_e_v_fit * offset_err_fit) ** 2 + (offset_fit * k_err_fit) ** 2)
                print('\n=======================================================')
                print('--- Step-by-Step Error Propagation ---')
                print('=======================================================')
                print('[Manual Input Method]')
                print(f'  V(ch) Slope : {fmt_err(slope_man, slope_err_man, 4)} mV/ch')
                print(f'  V(ch) Offset: {fmt_err(offset_man, offset_err_man, 2)} mV')
                print(f'  Source Peak : {fmt_err(source_channel_man, 1.0, 2)} ch (Assumed error)')
                print(f'  Source Voltg: {fmt_err(v_source_man, v_source_err_man, 2)} mV')
                print(f'  Scale   : {fmt_err(k_e_v_man, k_err_man, 5)} MeV/mV')
                print(f'  => E(ch) Slope : {fmt_err(slope_e_man, slope_e_err_man, 6)} MeV/ch')
                print(f'  => E(ch) Offset: {fmt_err(offset_e_man, offset_e_err_man, 6)} MeV')
                print('\n[Gaussian Fit Method]')
                print(f'  V(ch) Slope : {fmt_err(slope_fit, slope_err_fit, 4)} mV/ch')
                print(f'  V(ch) Offset: {fmt_err(offset_fit, offset_err_fit, 2)} mV')
                print(f'  Source Peak : {fmt_err(source_channel_fit, source_err_fit, 2)} ch')
                print(f'  Source Voltg: {fmt_err(v_source_fit, v_source_err_fit, 2)} mV')
                print(f'  Scale   : {fmt_err(k_e_v_fit, k_err_fit, 5)} MeV/mV')
                print(f'  => E(ch) Slope : {fmt_err(slope_e_fit, slope_e_err_fit, 6)} MeV/ch')
                print(f'  => E(ch) Offset: {fmt_err(offset_e_fit, offset_e_err_fit, 6)} MeV')
                ax3 = ax2.twinx()
                ax3.set_ylabel('Energy (MeV)', color='C1')
                ax3.plot(ch_line, slope_e_fit * ch_line + offset_e_fit, 'C1--', alpha=0.3)
                ax3.errorbar(source_channel_fit, 4.5, xerr=source_err_fit, fmt='r*', markersize=10, label='Source (4.5 MeV)')
                eq_text_e = f"E(MeV) = ({slope_e_fit:.5f} ± {slope_e_err_fit:.5f}) * ch {('+' if offset_e_fit >= 0 else '-')} {abs(offset_e_fit):.4f}"
                ax2.text(0.05, 0.95, eq_text_e, transform=ax2.transAxes, fontsize=10, color='C1', verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
                ax2.legend(loc='lower right')
                ax3.legend(loc='center right')
                plt.tight_layout()
                plt.show()
                experiment_state['calib_slope_e'] = slope_e_fit
                experiment_state['calib_slope_e_err'] = slope_e_err_fit
                experiment_state['calib_offset_e'] = offset_e_fit
                experiment_state['calib_offset_e_err'] = offset_e_err_fit
                t_sec = float(src_data.get('time_sec', 1.0))
                n_count = float(src_data.get('total_count', 0.0))
                d_slit = float(src_data.get('slit2', 6.0))
                L_dist = float(src_data.get('distance', 70.0))
                if L_dist > 0:
                    solid_angle_ratio = np.pi * (d_slit / 2.0) ** 2 / L_dist ** 2 / (4 * np.pi)
                    B_mes = n_count / t_sec if t_sec > 0 else 0.0
                    B_mes_err = np.sqrt(n_count) / t_sec if t_sec > 0 else 0.0
                    experiment_state['B_mes'] = B_mes
                    experiment_state['B_mes_err'] = B_mes_err
                    B_0_calc = B_mes / solid_angle_ratio
                    experiment_state['B_0'] = B_0_calc
                    n_count_err = np.sqrt(n_count) if n_count > 0 else 0.0
                    B_0_err = B_0_calc * (n_count_err / n_count) if n_count > 0 else 0.0
                    experiment_state['B_0_err'] = B_0_err
                    print('\n=======================================================')
                    print('--- Source Activity (B_0) Calculation ---')
                    print('=======================================================')
                    print(f'  Source Meta: Sensor Slit(s2)={d_slit} mm, Dist={L_dist} mm')
                    print(f'  Measurement Time (t): {t_sec} s')
                    print(f'  Total Counts (N)    : {n_count} ± {n_count_err:.2f}')
                    print(f'  Measured Rate (B_mes): {B_mes:.2f} ± {B_mes_err:.2e} cps')
                    print(f'  Solid Angle Ratio   : {solid_angle_ratio:.6e}')
                    print(f'  => Source Activity (B_0): {B_0_calc:.2e} ± {B_0_err:.2e} Bq')
            else:
                ax2.legend(loc='lower right')
                plt.tight_layout()
                plt.show()
            print('\n=======================================================')
            print('1. Raw Pulser Data')
            print('=======================================================')
            print(df_res.to_string(index=False))
            comp_data = []
            err_man_mev = float(1.0 * slope_man * k_e_v_man) if source_channel_man > 0 else np.nan
            comp_data.append({'Method': 'Manual Input', 'Slope (mV/ch)': fmt_err(slope_man, slope_err_man, 4), 'Offset (mV)': fmt_err(offset_man, offset_err_man, 2), 'Source Peak (ch)': fmt_err(source_channel_man, 1.0, 2) if source_channel_man > 0 else 'N/A', 'Error at 4.5MeV (MeV)': f'± {abs(err_man_mev):.5f}' if not np.isnan(err_man_mev) else 'N/A'})
            err_fit_mev = float(source_err_fit * slope_fit * (4.5 / v_source_fit)) if has_source and source_channel_fit > 0 else np.nan
            comp_data.append({'Method': 'Gaussian Fit', 'Slope (mV/ch)': fmt_err(slope_fit, slope_err_fit, 4), 'Offset (mV)': fmt_err(offset_fit, offset_err_fit, 2), 'Source Peak (ch)': fmt_err(source_channel_fit, source_err_fit, 2) if source_channel_fit > 0 else 'N/A', 'Error at 4.5MeV (MeV)': f'± {abs(err_fit_mev):.5f}' if not np.isnan(err_fit_mev) else 'N/A'})
            print('\n=======================================================')
            print('2. Precision Comparison: Manual vs Gaussian Fit')
            print('=======================================================')
            df_comp = pd.DataFrame(comp_data)
            print(df_comp.to_string(index=False))
            print('\n--- Calibration Process Completed Successfully ---')
        except Exception as e:
            print(f'\nCRITICAL ERROR: {e}')
            traceback.print_exc()
display(out_calib)
if linked:
    run_calibration(None)


In [ ]:
# @title 2.1 薄膜の厚さ測定
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import ipywidgets as widgets
from IPython.display import display, clear_output
import traceback

def get_clean_data(f_dict):
    raw = f_dict.get('raw_data', [])
    if not raw:
        return np.array([])
    if len(raw) > 5:
        try:
            float(raw[-1])
        except:
            raw = raw[:-5]
    cleaned = [float(v) for v in raw if str(v).replace('.', '', 1).isdigit()]
    return np.array(cleaned[:1024])

def gaussian(x, a, mu, sigma):
    return a * np.exp(-(x - mu) ** 2 / (2 * sigma ** 2))

def calc_thickness_linear(E_rem, E_rem_err, a, b_intercept, cov_ab):
    """Calculates thickness d = R(4.5) - R(E_rem) for linear fit R(E) = a*E + b."""
    d = a * (4.5 - E_rem)
    dd_dE = -a
    dd_da = 4.5 - E_rem
    dd_db = 0.0
    J_ab = np.array([dd_da, dd_db])
    var_ab = J_ab.T @ cov_ab @ J_ab if cov_ab is not None else 0.0
    var_E = (dd_dE * E_rem_err) ** 2
    d_err = np.sqrt(var_E + var_ab)
    return (d, d_err)
out_thick = widgets.Output()
energies = [1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.1, 4.5]
au_inputs = [SimpleNamespace(value=v) for v in [2.699, 3.435, 4.23, 5.086, 6.002, 6.974, 7.175, 8.001]]
al_inputs = [SimpleNamespace(value=v) for v in [5.017, 6.804, 8.777, 10.944, 13.308, 15.867, 16.402, 18.618]]

def run_thickness(b):
    require_link()
    with out_thick:
        clear_output()
        print('--- Thickness Calculation with Full Error Propagation ---')
        try:
            if 'cal_slope' not in experiment_state:
                print('ERROR: Calibration not completed. Run Block 1 first.')
                return
            thick_files = [f for f in experiment_state['uploaded_files'].values() if f['type'] == 'Thickness']
            e_ref_au = np.array([energies[i] for i in range(len(energies)) if au_inputs[i].value > 0])
            r_ref_au = np.array([au_inputs[i].value for i in range(len(energies)) if au_inputs[i].value > 0])
            e_ref_al = np.array([energies[i] for i in range(len(energies)) if al_inputs[i].value > 0])
            r_ref_al = np.array([al_inputs[i].value for i in range(len(energies)) if al_inputs[i].value > 0])

            def range_func_linear(E, a, b):
                return a * E + b
            popt_au, cov_au, popt_al, cov_al = (None, None, None, None)
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
            e_line = np.linspace(0.5, 5.0, 100)
            if len(e_ref_au) >= 2:
                popt_au, cov_au = curve_fit(range_func_linear, e_ref_au, r_ref_au, p0=[1.0, 0.0])
                ax2.plot(e_ref_au, r_ref_au, 'o', color='goldenrod', label='Au Ref Data')
                ax2.plot(e_line, range_func_linear(e_line, *popt_au), '--', color='goldenrod', label=f"Au Fit: {popt_au[0]:.2f}E{('+' if popt_au[1] >= 0 else '')}{popt_au[1]:.2f}")
            if len(e_ref_al) >= 2:
                popt_al, cov_al = curve_fit(range_func_linear, e_ref_al, r_ref_al, p0=[1.0, 0.0])
                ax2.plot(e_ref_al, r_ref_al, 'o', color='steelblue', label='Al Ref Data')
                ax2.plot(e_line, range_func_linear(e_line, *popt_al), '--', color='steelblue', label=f"Al Fit: {popt_al[0]:.2f}E{('+' if popt_al[1] >= 0 else '')}{popt_al[1]:.2f}")
            ax2.set_title('Alpha Range vs Energy (Linear Fit)')
            ax2.set_xlabel('Energy (MeV)')
            ax2.set_ylabel('Range (um)')
            ax1.set_title('Energy Loss Spectra')
            ax1.set_xlabel('Energy (MeV)')
            ax1.set_ylabel('Counts')
            slope_e = experiment_state['cal_slope']
            slope_e_err = experiment_state['cal_slope_err']
            offset_e = experiment_state['cal_offset']
            offset_e_err = experiment_state['cal_offset_err']
            for f in thick_files:
                y_data = get_clean_data(f)
                if len(y_data) == 0:
                    continue
                mat = f.get('material', '')
                x_ch = np.arange(len(y_data))
                mu_guess_ch, a_guess = (np.argmax(y_data), np.max(y_data))
                try:
                    popt_s, pcov_s = curve_fit(gaussian, x_ch, y_data, p0=[a_guess, mu_guess_ch, 5.0])
                    ch_rem = popt_s[1]
                    ch_rem_err = np.sqrt(np.diag(pcov_s))[1]
                    e_rem = slope_e * ch_rem + offset_e
                    e_rem_err = np.sqrt((slope_e * ch_rem_err) ** 2 + (ch_rem * slope_e_err) ** 2 + offset_e_err ** 2)
                    x_energy = x_ch * slope_e + offset_e
                    plot_color = 'goldenrod' if mat == 'Au' else 'steelblue'
                    ax1.plot(x_energy, y_data, alpha=0.3, color=plot_color, linestyle='-')
                    ax1.plot(x_energy, gaussian(x_ch, *popt_s), color=plot_color, linestyle='--', label=f'{mat} {e_rem:.2f}MeV')
                    if mat == 'Au' and popt_au is not None:
                        thickness, thick_err = calc_thickness_linear(e_rem, e_rem_err, popt_au[0], popt_au[1], cov_au)
                        experiment_state['d_Au'] = thickness
                        experiment_state['d_Au_err'] = thick_err
                    elif mat == 'Al' and popt_al is not None:
                        thickness, thick_err = calc_thickness_linear(e_rem, e_rem_err, popt_al[0], popt_al[1], cov_al)
                        experiment_state['d_Al'] = thickness
                        experiment_state['d_Al_err'] = thick_err
                    else:
                        continue
                    print(f"\n--- {mat} Target ({f['file_name']}) ---")
                    print(f'Peak Channel: {ch_rem:.2f} ± {ch_rem_err:.2f} ch (Stat Error)')
                    print(f'Remaining E : {e_rem:.4f} ± {e_rem_err:.4f} MeV (Total Propagated)')
                    print(f'Thickness d : {thickness:.3f} ± {thick_err:.3f} um')
                    range_rem = range_func_linear(e_rem, popt_au[0], popt_au[1]) if mat == 'Au' else range_func_linear(e_rem, popt_al[0], popt_al[1])
                    ax2.plot(e_rem, range_rem, '*', color='red', markersize=10)
                    ax2.annotate(f'{mat}', (e_rem, range_rem), xytext=(5, 5), textcoords='offset points', color='red')
                except Exception as e:
                    print(f"WARNING: Failed processing {f['file_name']}. Error: {e}")
            if ax1.get_legend_handles_labels()[0]:
                ax1.legend()
            if ax2.get_legend_handles_labels()[0]:
                ax2.legend()
            plt.tight_layout()
            plt.show()
        except Exception as err:
            print(f'\nCRITICAL ERROR: {err}')
            traceback.print_exc()
display(out_thick)
if linked:
    run_thickness(None)


In [ ]:
# @title 5.1 散乱軌道と原子核サイズのシミュレーション
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
sld_theta = widgets.FloatSlider(value=15, min=10, max=170, step=1, description='Theta (deg):', layout=widgets.Layout(width='500px'), continuous_update=True)
chk_fixed_scale = widgets.Checkbox(value=True, description='Fix axis scale', layout=widgets.Layout(width='300px'))
out_sim = widgets.Output()

def get_limit(theta_deg, k_c, R_alpha):
    theta = np.radians(theta_deg)
    a = k_c * (1.0 + 1.0 / np.sin(theta / 2.0))
    b = k_c / np.tan(theta / 2.0)
    R_max = max(0.0, a - R_alpha)
    return max(a, b) * 1.5 + R_max

def update_sim(change):
    if not linked:
        return
    theta_deg = sld_theta.value
    theta = np.radians(theta_deg)
    E_alpha = 4.5
    Z_Au = 79.0
    Z_Al = 13.0
    R_alpha = 1.2 * 4 ** (1 / 3)
    k_c_Au = 1.44 * Z_Au / E_alpha
    k_c_Al = 1.44 * Z_Al / E_alpha
    a_Au = k_c_Au * (1.0 + 1.0 / np.sin(theta / 2.0))
    b_Au = k_c_Au / np.tan(theta / 2.0)
    a_Al = k_c_Al * (1.0 + 1.0 / np.sin(theta / 2.0))
    b_Al = k_c_Al / np.tan(theta / 2.0)
    R_max_Au = max(0.0, a_Au - R_alpha)
    R_max_Al = max(0.0, a_Al - R_alpha)
    R_th_Au = 1.2 * 197 ** (1 / 3)
    R_th_Al = 1.2 * 27 ** (1 / 3)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6.5))

    def draw_subplot(ax, title, Z, a, b, k_c, R_max, R_th, color):
        ax.set_title(title)
        ax.set_aspect('equal')
        ax.set_xlabel('x (fm)')
        ax.set_ylabel('y (fm)')
        ax.plot(0, 0, 'ko', markersize=6, zorder=5)
        circle_th = plt.Circle((0, 0), R_th, color=color, alpha=0.4, label=f'Theoretical Nucleus\n(R={R_th:.1f} fm)', zorder=2)
        ax.add_patch(circle_th)
        circle_max = plt.Circle((0, 0), R_max, color='red', fill=False, linestyle='--', linewidth=1.5, label=f'Upper Limit\n(R_max={R_max:.1f} fm)', zorder=3)
        ax.add_patch(circle_max)
        phi = np.pi / 2.0 - theta / 2.0
        c_h = k_c / np.sin(theta / 2.0)
        t = np.linspace(-4, 4, 300)
        x0 = k_c * np.cosh(t) + c_h
        y0 = b * np.sinh(t)
        x1 = x0 * np.cos(phi) - y0 * np.sin(phi)
        y1 = x0 * np.sin(phi) + y0 * np.cos(phi)
        ax.plot(-x1, y1, 'k-', linewidth=2, label='Alpha Trajectory', zorder=4)
        x_cl = -((k_c + c_h) * np.cos(phi))
        y_cl = (k_c + c_h) * np.sin(phi)
        ax.plot([0, x_cl], [0, y_cl], 'k:', linewidth=1.5, zorder=3)
        alpha_particle = plt.Circle((x_cl, y_cl), R_alpha, color='blue', alpha=0.5, label=f'Alpha Particle\n(R={R_alpha:.1f} fm)', zorder=5)
        ax.add_patch(alpha_particle)
        if chk_fixed_scale.value:
            limit = get_limit(10.0, k_c, R_alpha)
        else:
            limit = max(a, b) * 1.5 + R_max
        line_len = limit * 2.0
        ax.plot([-line_len, line_len], [0, 0], color='gray', linestyle=':', linewidth=1, zorder=1)
        ax.plot([0, line_len * np.cos(theta)], [0, line_len * np.sin(theta)], color='gray', linestyle=':', linewidth=1, zorder=1)
        ax.plot([-line_len, -k_c], [b, b], 'g--', alpha=0.7, linewidth=1.5, label='Asymptotes')
        x_out = np.array([-k_c, -k_c + line_len * np.cos(theta)])
        y_out = np.array([b, b + line_len * np.sin(theta)])
        ax.plot(x_out, y_out, 'g--', alpha=0.7, linewidth=1.5)
        ax.set_xlim(-limit * 0.8, limit * 1.2)
        ax.set_ylim(-limit * 0.5, limit * 1.2)
        ax.grid(True, linestyle=':', alpha=0.5)
        ax.legend(loc='upper right', fontsize=8)
    draw_subplot(ax1, 'Gold (Au) Target', Z_Au, a_Au, b_Au, k_c_Au, R_max_Au, R_th_Au, 'goldenrod')
    draw_subplot(ax2, 'Aluminum (Al) Target', Z_Al, a_Al, b_Al, k_c_Al, R_max_Al, R_th_Al, 'steelblue')
    fig.suptitle(model_caption())
    plt.tight_layout()
    with out_sim:
        clear_output(wait=True)
        print(f'Radius limits at scattering angle {theta_deg:.1f} deg')
        print(f'[Gold (Au)] Closest approach a: {a_Au:.2f} fm | Nuclear radius upper limit R_max: {R_max_Au:.2f} fm (reference: ~{R_th_Au:.2f} fm)')
        print(f'[Aluminum (Al)] Closest approach a: {a_Al:.2f} fm | Nuclear radius upper limit R_max: {R_max_Al:.2f} fm (reference: ~{R_th_Al:.2f} fm)')
        display(fig)
    plt.close(fig)

def render_trajectory(_=None):
    require_link()
    value = active_model_record.get('metric_a') if active_model_record else None
    try:
        value = float(value)
    except (TypeError, ValueError):
        value = 0.0
    if not np.isfinite(value) or not 10 <= value <= 180:
        angles = [float(f['angle']) for f in experiment_state['uploaded_files'].values() if f.get('type') == 'Measurement' and f.get('angle') not in ('', None)]
        value = max(angles) if angles else 15.0
    sld_theta.unobserve(update_sim, names='value')
    try:
        sld_theta.max = max(170, value)
        sld_theta.value = value
    finally:
        sld_theta.observe(update_sim, names='value')
    update_sim(None)
sld_theta.observe(update_sim, names='value')
chk_fixed_scale.observe(update_sim, names='value')
display(widgets.VBox([widgets.HBox([sld_theta, chk_fixed_scale]), out_sim]))
if linked:
    render_trajectory()


In [ ]:
# @title 6.1 幾何学的限界の視覚化
out_geom = widgets.Output()
inp_theta = widgets.FloatText(value=60.0, description='Theta (deg):')
drop_s1 = widgets.Dropdown(options=[2.0, 4.0, 6.0], value=4.0, description='Slit 1 (mm):')
drop_s2 = widgets.Dropdown(options=[2.0, 4.0, 6.0], value=6.0, description='Slit 2 (mm):')
drop_L = widgets.Dropdown(options=[30.0, 50.0, 70.0], value=70.0, description='Dist L (mm):')

def update_geometry(change=None):
    if not linked:
        return
    with out_geom:
        clear_output(wait=True)
        theta = inp_theta.value
        s1 = drop_s1.value
        s2 = drop_s2.value
        L = drop_L.value
        if not 15.0 <= theta <= 180.0:
            print('ERROR: Angle must be between 15 and 180 degrees.')
            return
        t_min, t_max, Tp_min, Sp_min, Dp_min, Tp_max, Sp_max, Dp_max = calculate_angle_limits(theta, s1, s2, L)
        L_s1 = 11.7
        L3 = 60.0
        r_src = 5.0
        dr1 = s1 / 2.0
        u_x, u_y = (np.cos(np.pi / 6), np.sin(np.pi / 6))
        SL = np.array([-r_src, -L])
        SR = np.array([r_src, -L])
        S1L = np.array([-dr1 * u_x, -L_s1 - dr1 * u_y])
        S1R = np.array([dr1 * u_x, -L_s1 + dr1 * u_y])

        def get_crossed_beam_x_range(y_target):
            if abs(S1R[1] - SL[1]) > 1e-06 and abs(S1L[1] - SR[1]) > 1e-06:
                x_left = SL[0] + (y_target - SL[1]) * (S1R[0] - SL[0]) / (S1R[1] - SL[1])
                x_right = SR[0] + (y_target - SR[1]) * (S1L[0] - SR[0]) / (S1L[1] - SR[1])
            else:
                x_left, x_right = (-r_src, r_src)
            return (min(x_left, x_right), max(x_left, x_right))
        theta_rad = np.radians(theta)
        Dc = np.array([-L3 * np.sin(theta_rad), L3 * np.cos(theta_rad)])
        dD = np.array([s2 / 2.0 * np.cos(theta_rad), s2 / 2.0 * np.sin(theta_rad)])
        direct_hit = False
        for t in np.linspace(-1, 1, 100):
            p = Dc + dD * t
            px, py = (p[0], p[1])
            if py > -L_s1:
                xl, xr = get_crossed_beam_x_range(py)
                if xl <= px <= xr:
                    direct_hit = True
                    break
        sin_nom = np.sin(np.radians(theta / 2.0))
        ratio_min = (sin_nom / np.sin(np.radians(t_min / 2.0))) ** 4 * 100.0 if t_min > 0 else float('inf')
        ratio_max = (sin_nom / np.sin(np.radians(t_max / 2.0))) ** 4 * 100.0 if t_max > 0 else 0.0
        print(f'=== Geometric Angular Limits ===')
        print(f'Nominal Angle (Theta) : {theta:.1f}°')
        print(f'Minimum Angle         : {t_min:.2f}° (Effective Rate: {ratio_min:.1f}%)')
        print(f'Maximum Angle         : {t_max:.2f}° (Effective Rate: {ratio_max:.1f}%)')
        print(f'Angular Uncertainty   : ±{(t_max - t_min) / 2:.2f}°')
        if direct_hit:
            print(f'Direct Beam Inflow    : [WARNING] Direct beam hits the detector!')
        else:
            print(f'Direct Beam Inflow    : Safe (No direct beam)')
        print('\n')
        fig, ax = plt.subplots(figsize=(9, 9))
        y_steps = np.linspace(-L, L3 * 1.3, 50)
        x_lefts, x_rights = ([], [])
        for y_t in y_steps:
            xl, xr = get_crossed_beam_x_range(y_t)
            x_lefts.append(xl)
            x_rights.append(xr)
        X_beam = list(x_lefts) + list(reversed(x_rights))
        Y_beam = list(y_steps) + list(reversed(y_steps))
        ax.fill(X_beam, Y_beam, color='orange', alpha=0.2, label='Direct Beam Area', zorder=1)
        target_len = 15.0
        ax.plot([-target_len * u_x, target_len * u_x], [-target_len * u_y, target_len * u_y], color='gold', linewidth=3, label='Target Surface (30°)', zorder=3)
        ax.plot(0, 0, 'ko', markersize=6, label='Target Center', zorder=4)
        ax.plot([-r_src, r_src], [-L, -L], 'b-', linewidth=5, label='Source Window (r=5mm)', zorder=3)
        ax.plot([S1L[0], S1R[0]], [S1L[1], S1R[1]], 'c-', linewidth=4, label=f'Slit 1 ({s1}mm, 30°)', zorder=3)
        sensor_color = 'red' if direct_hit else 'green'
        ax.plot([Dc[0] - dD[0], Dc[0] + dD[0]], [Dc[1] - dD[1], Dc[1] + dD[1]], color=sensor_color, linewidth=5, label=f'Slit 2 ({s2}mm)', zorder=4)
        ax.plot([0, 0], [-L, 0], 'k--', alpha=0.3)
        ax.plot([0, Dc[0]], [0, Dc[1]], 'k--', alpha=0.3, label=f'Nominal Trajectory ({theta:.1f}°)')
        if Tp_max is not None:
            ax.plot([Sp_max[0], Tp_max[0]], [Sp_max[1], Tp_max[1]], 'g-', alpha=0.8, zorder=2)
            ax.plot([Tp_max[0], Dp_max[0]], [Tp_max[1], Dp_max[1]], 'g-', alpha=0.8, zorder=2)
            ax.lines[-1].set_label(f'Max Angle ({t_max:.1f}°)')
        if Tp_min is not None:
            ax.plot([Sp_min[0], Tp_min[0]], [Sp_min[1], Tp_min[1]], 'm-', alpha=0.8, zorder=2)
            ax.plot([Tp_min[0], Dp_min[0]], [Tp_min[1], Dp_min[1]], 'm-', alpha=0.8, zorder=2)
            ax.lines[-1].set_label(f'Min Angle ({t_min:.1f}°)')
        ax.set_aspect('equal')
        ax.set_title('Scattering Geometry & Direct Beam Acceptance (Top View)')
        ax.set_xlabel('x (mm)')
        ax.set_ylabel('y (mm)')
        ax.legend(loc='upper left', bbox_to_anchor=(1.05, 1))
        ax.grid(True, linestyle=':', alpha=0.6)
        ax.set_xlim(-L3 * 1.5, L3 * 1.5)
        ax.set_ylim(-L * 1.1, L3 * 1.2)
        plt.tight_layout()
        plt.show()

def render_geometry(_=None):
    require_link()
    measurements = [f for f in experiment_state['uploaded_files'].values() if f.get('type') == 'Measurement' and 15 <= float(f.get('angle', 0)) <= 180]
    if measurements:
        f = measurements[0]
        inp_theta.value = float(f['angle'])
        drop_s1.value = float(f['slit1'])
        drop_s2.value = float(f['slit2'])
        drop_L.value = float(f['distance'])
    update_geometry()
for control in (inp_theta, drop_s1, drop_s2, drop_L):
    control.observe(update_geometry, names='value')
display(widgets.VBox([widgets.HBox([inp_theta, drop_L]), widgets.HBox([drop_s1, drop_s2]), out_geom]))
if linked:
    render_geometry()


In [ ]:
# @title 8.1 個人モデルの更新
def predict_counts_with_error(theta_deg, time, s1, s2, L, isAu):
    """Return (expected counts, statistical error, systematic + error, systematic - error)."""
    if not 15.0 <= theta_deg <= 180.0:
        raise ValueError('Angle must be between 15 and 180 degrees.')
    e2_fm = 197.3 / 137.036
    z_alpha = 2.0
    Z_target = 79.0 if isAu else 13.0
    theta_rad = np.deg2rad(theta_deg)
    E_kin = 4.5
    B_0 = experiment_state.get('B_0', 0.0)
    B_0_err = experiment_state.get('B_0_err', 0.0)
    if isAu:
        n_target = experiment_state.get('n_Au', 0.0)
        n_err = experiment_state.get('n_Au_err', 0.0)
    else:
        n_target = experiment_state.get('n_Al', 0.0)
        n_err = experiment_state.get('n_Al_err', 0.0)
    n_target *= 1e-26
    n_err *= 1e-26
    omega_1 = np.pi * (s1 / 2 / (L - 11.7)) ** 2
    omega_2 = np.pi * (s2 / 2 / 60) ** 2
    if isAu:
        a_fm = z_alpha * Z_target * e2_fm / (4.0 * E_kin)
    else:
        a_fm = z_alpha * Z_target * e2_fm / (4.0 * E_kin)
    t_min, t_max, _, _, _, _, _, _ = calculate_angle_limits(theta_deg, s1, s2, L)
    theta_rad = np.radians(theta_deg)
    t_rad_min = np.radians(max(0.1, t_min))
    t_rad_max = np.radians(min(179.9, t_max))
    dsigma_domega = a_fm ** 2 / np.sin(theta_rad / 2.0) ** 4
    ds_min = a_fm ** 2 / np.sin(t_rad_max / 2.0) ** 4
    ds_max = a_fm ** 2 / np.sin(t_rad_min / 2.0) ** 4
    incident_rate = B_0 * (omega_1 / (4.0 * np.pi))
    expected_counts = incident_rate * n_target * dsigma_domega * omega_2 * time
    geometry_err_plus = incident_rate * n_target * abs(ds_max - dsigma_domega) * omega_2 * time
    geometry_err_minus = incident_rate * n_target * abs(ds_min - dsigma_domega) * omega_2 * time
    count_factor = omega_1 / (4.0 * np.pi) * dsigma_domega * omega_2 * time
    propagated_variance = (count_factor * n_target * B_0_err) ** 2 + (count_factor * B_0 * n_err) ** 2
    sys_err_plus = np.sqrt(propagated_variance + geometry_err_plus ** 2)
    sys_err_minus = np.sqrt(propagated_variance + geometry_err_minus ** 2)
    stat_err = np.sqrt(expected_counts) if expected_counts > 0 else 0.0
    return (expected_counts, stat_err, sys_err_plus, sys_err_minus)

# Edit the function above, run this cell, then run 8.2 to save.
# Keeping the example unchanged preserves the model loaded in 0.1.
activate_model_definition(predict_counts_with_error)


In [ ]:
# @title 8.2 モデルの評価と可視化・データポイント詳細表
out_model = widgets.Output()

def run_model_evaluation(b=None, upload=False):
    global active_model_record, active_model_kind
    require_link()
    with out_model:
        clear_output()
        print('--- Model Evaluation Started ---')
        if 'experiment_state' not in globals():
            print('[ERROR] experiment_state is missing. Please run Block 0.1 first.')
            return
        group_id = experiment_state.get('group_id', '')
        pin = experiment_state.get('pin', '')
        gas_url = experiment_state.get('gas_url', '')
        if not group_id or not gas_url or gas_url == 'YOUR_GAS_WEB_APP_URL_HERE':
            print('[ERROR] Group ID or GAS URL is not configured.')
            return
        user_predict_func = globals().get('predict_counts_with_error')
        if not callable(user_predict_func):
            print("[ERROR] 'predict_counts_with_error' is not defined. Please run Block 8.1 first.")
            return
        teacher_predict_func = globals().get('local_teacher_predict')
        if not callable(teacher_predict_func):
            print("[ERROR] 'local_teacher_predict' is not defined. Please run the Teacher Model block first.")
            return
        all_files = server_data_cache['all_files']
        all_models = server_data_cache['all_models']
        try:
            background_count_estimate(0.0)
        except ValueError as e:
            print(f'[ERROR] {e}')
            return
        B_0 = experiment_state.get('B_0', 2000000.0)
        my_meas_files = [f for f in all_files if f.get('group_id') == group_id and f.get('type') == 'Measurement']
        if not my_meas_files:
            my_meas_files = [f for f in experiment_state.get('uploaded_files', {}).values() if f.get('type') == 'Measurement']
        if not my_meas_files:
            print('[WARNING] No measurement data found for your group.')
            return
        parsed_data = []
        for f in my_meas_files:
            mat = f.get('material', 'Au')
            isAu = mat == 'Au'
            theta = float(f.get('angle', 0.0))
            t_sec = float(f.get('time_sec', 1.0))
            total_cnt = measurement_count(f)
            s1 = float(f.get('slit1', 4.0))
            s2 = float(f.get('slit2', 6.0))
            L = float(f.get('distance', 70.0))
            bg_expected, bg_stat_err = background_count_estimate(t_sec)
            try:
                net_cnt, net_cnt_err_up, net_cnt_err_down = observation_with_dead_time(total_cnt, t_sec)
            except ValueError as e:
                print(f'[WARNING] Excluded {mat} at {theta} deg: {e}')
                continue
            obs_stat_err = np.sqrt(total_cnt)
            L_s1 = L - 11.7 if L - 11.7 > 0 else 1.0
            omega_1 = np.pi * (s1 / 2.0) ** 2 * np.cos(np.radians(30.0)) / L_s1 ** 2
            omega_2 = np.pi * (s2 / 2.0) ** 2 / 60.0 ** 2
            n_target = experiment_state.get('n_Au' if isAu else 'n_Al', 1e+18)
            incident_rate = B_0 * (omega_1 / (4.0 * np.pi))
            denom = incident_rate * n_target * omega_2 * t_sec
            if not 15.0 <= theta <= 180.0:
                print(f'[WARNING] Excluded {mat} at {theta} deg: angle outside 15-180 degrees.')
                continue
            user_result = user_predict_func(theta, t_sec, s1, s2, L, isAu)
            teacher_result = teacher_predict_func(theta, t_sec, s1, s2, L, isAu)
            if not valid_prediction(user_result) or not valid_prediction(teacher_result) or (not np.isfinite(denom)) or (denom <= 0) or (not np.all(np.isfinite([net_cnt, net_cnt_err_up, net_cnt_err_down]))):
                print(f'[WARNING] Excluded {mat} at {theta} deg: invalid value or uncertainty.')
                continue
            ds_meas_cm2 = net_cnt / denom if denom > 0 else 0.0
            ds_meas_err_cm2_up = net_cnt_err_up / denom if denom > 0 else 0.0
            ds_meas_err_cm2_down = net_cnt_err_down / denom if denom > 0 else 0.0
            ds_meas = ds_meas_cm2 * 1e+26
            ds_meas_err_up = ds_meas_err_cm2_up * 1e+26
            ds_meas_err_down = ds_meas_err_cm2_down * 1e+26
            parsed_data.append({'material': mat, 'isAu': isAu, 'angle': theta, 's1': s1, 's2': s2, 'L': L, 't_sec': t_sec, 'total_cnt': total_cnt, 'obs_stat_err': obs_stat_err, 'bg_expected': bg_expected, 'bg_stat_err': bg_stat_err, 'net_cnt': net_cnt, 'net_cnt_err_up': net_cnt_err_up, 'net_cnt_err_down': net_cnt_err_down, 'denom': denom, 'ds_meas': ds_meas, 'ds_meas_err_up': ds_meas_err_up, 'ds_meas_err_down': ds_meas_err_down, 'pull': 0.0, 'll_diff': 0.0})
        if not parsed_data:
            print('[WARNING] No valid data remain for evaluation or plotting.')
            return
        df_my = pd.DataFrame(parsed_data).sort_values('angle')
        table_rows = []
        valid_angles = []
        log_likelihood_diff = 0.0
        for idx, row in df_my.iterrows():
            th = row['angle']
            isAu = row['isAu']
            p_cnt, stat_p_cnt, sys_eu_cnt, sys_el_cnt = user_predict_func(th, row['t_sec'], row['s1'], row['s2'], row['L'], isAu)
            denom = row['denom']
            p_ds = p_cnt / denom * 1e+26 if denom > 0 else 0.0
            eu_ds = sys_eu_cnt / denom * 1e+26 if denom > 0 else 0.0
            el_ds = sys_el_cnt / denom * 1e+26 if denom > 0 else 0.0
            m_ds = row['ds_meas']
            m_err_up_ds = row['ds_meas_err_up']
            m_err_down_ds = row['ds_meas_err_down']
            t_cnt, stat_t_cnt, tu_cnt, tl_cnt = teacher_predict_func(th, row['t_sec'], row['s1'], row['s2'], row['L'], isAu)
            t_ds = t_cnt / denom * 1e+26 if denom > 0 else 0.0
            stat_t_ds = stat_t_cnt / denom * 1e+26 if denom > 0 else 0.0
            tu_ds = tu_cnt / denom * 1e+26 if denom > 0 else 0.0
            tl_ds = tl_cnt / denom * 1e+26 if denom > 0 else 0.0
            pull, ll_diff, chi2_user, log_var_user, chi2_teach, log_var_teach = compare_predictions(m_ds, m_err_up_ds, m_err_down_ds, (p_ds, stat_p_cnt / denom * 1e+26, eu_ds, el_ds), (t_ds, stat_t_ds, tu_ds, tl_ds))
            chi2_diff = chi2_teach - chi2_user
            log_var_diff = log_var_teach - log_var_user
            if abs(pull) < 1.0:
                valid_angles.append(th)
            df_my.at[idx, 'pull'] = pull
            df_my.at[idx, 'll_diff'] = ll_diff
            table_rows.append({'Target': row['material'], 'Angle(deg)': f'{int(th)}', 's1/s2/L(mm)': f"{int(row['s1'])}/{int(row['s2'])}/{int(row['L'])}", 'Time(min)': f"{int(row['t_sec'] / 60.0)}", 'Obs Cnt': f"{int(row['total_cnt'])}", 'Obs stat err': f"{row['obs_stat_err']:.1f}", 'BG Cnt': f"{row['bg_expected']:.1f} ±{row['bg_stat_err']:.1f}", 'Corrected Net Signal Cnt': f"{row['net_cnt']:.1f} ±{row['net_cnt_err_up']:.1f}", 'Predicted Cnt': f'{p_cnt:.1f} ±{stat_p_cnt:.1f}(stat) +{sys_eu_cnt:.1f}/-{sys_el_cnt:.1f}(syst)', 'Pull': f'{pull:.2e}', 'log(L_s/L_0)': f'{ll_diff:.2e}', '(x-mu)^2/sigma^2 (s)': f'{chi2_user:.2e}', '(x-mu)^2/sigma^2 (0)': f'{chi2_teach:.2e}', 'Diff (x-mu)^2/sigma^2': f'{chi2_diff:.2e}', 'log(2pisigma^2) (s)': f'{log_var_user:.2e}', 'log(2pisigma^2) (0)': f'{log_var_teach:.2e}', 'Diff log(2pisigma^2)': f'{log_var_diff:.2e}'})
        metric_a = max(valid_angles) if valid_angles else 0.0
        for idx, row in df_my.iterrows():
            th = row['angle']
            if 'calculate_angle_limits' in globals():
                t_min, _, _, _, _, _, _, _ = calculate_angle_limits(th, row['s1'], row['s2'], row['L'])
            else:
                t_min = th - 5.0
            if t_min > 0.0 and th <= metric_a:
                log_likelihood_diff += row['ll_diff']
        score_b = log_likelihood_diff
        evaluated_metrics = (float(metric_a), float(score_b))
        if upload:
            upload_model_code = model_code_with_feedback(active_model_code, checkbox_feedback.value)
            api_call('upload_metrics_and_model', metric_a=float(metric_a), score_b=float(score_b), model_code=upload_model_code)
            active_model_record = {'group_id': experiment_state['personal_id'], 'metric_a': float(metric_a), 'score_b': float(score_b), 'model_code': upload_model_code}
            active_model_kind = 'Personal'
            all_models[:] = [m for m in all_models if str(m.get('group_id', '')) != experiment_state['personal_id']] + [dict(active_model_record)]
            print(f"Saved personal model and metrics as {experiment_state['personal_id']}.")
        elif active_model_record is not None:
            try:
                saved_a = float(active_model_record['metric_a'])
                saved_b = float(active_model_record['score_b'])
                if np.all(np.isfinite([saved_a, saved_b])):
                    metric_a, score_b = (saved_a, saved_b)
            except (TypeError, ValueError, KeyError):
                pass
        print(model_caption())
        print(f'Recalculated metrics: a={evaluated_metrics[0]:.1f}, b={evaluated_metrics[1]:.4f}')
        print('=== Active Model Metrics ===')
        print(f'Metric a (Max angle with |pull| < 1) : {metric_a:.1f} deg')
        print(f'Metric b (Log-Likelihood Ratio log(L_s / L_0)) : {score_b:.4f}')
        print('\n--- Detailed Data Points Table ---')
        df_display = pd.DataFrame(table_rows)
        display(df_display)
        theta_grid = np.linspace(15, 120, 200)
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 12), sharex=True, gridspec_kw={'height_ratios': [3, 1]})
        colors = {'Au': 'goldenrod', 'Al': 'deepskyblue'}
        exclude_group_ids = ['test', 'ta']
        plot_t_sec = 600.0
        plot_s1 = 6.0
        plot_s2 = 6.0
        plot_L = 70.0
        plot_L_s1 = plot_L - 11.7 if plot_L - 11.7 > 0 else 1.0
        plot_omega_1 = np.pi * (plot_s1 / 2.0) ** 2 * np.cos(np.radians(30.0)) / plot_L_s1 ** 2
        plot_omega_2 = np.pi * (plot_s2 / 2.0) ** 2 / 60.0 ** 2
        plot_incident_rate = B_0 * (plot_omega_1 / (4.0 * np.pi))
        for mat, isAu in [('Au', True), ('Al', False)]:
            c = colors[mat]
            n_target_plot = experiment_state.get('n_Au' if isAu else 'n_Al', 1e+18)
            plot_denom = plot_incident_rate * n_target_plot * plot_omega_2 * plot_t_sec
            vals = []
            t_vals = []
            for th in theta_grid:
                p_cnt, stat_p_cnt, sys_eu_cnt, sys_el_cnt = user_predict_func(th, plot_t_sec, plot_s1, plot_s2, plot_L, isAu)
                teacher_result = teacher_predict_func(th, plot_t_sec, plot_s1, plot_s2, plot_L, isAu)
                t_cnt = teacher_result[0]
                if not valid_prediction((p_cnt, stat_p_cnt, sys_eu_cnt, sys_el_cnt)) or not valid_prediction(teacher_result) or (not np.isfinite(plot_denom)) or (plot_denom <= 0):
                    vals.append((th, np.nan, np.nan, np.nan))
                    t_vals.append((th, np.nan))
                    print(f'[WARNING] Excluded {mat} model curve at {th:.2f} deg: invalid uncertainty.')
                    continue
                p_ds = p_cnt / plot_denom * 1e+26 if plot_denom > 0 else 0.0
                eu_ds = sys_eu_cnt / plot_denom * 1e+26 if plot_denom > 0 else 0.0
                el_ds = sys_el_cnt / plot_denom * 1e+26 if plot_denom > 0 else 0.0
                t_ds = t_cnt / plot_denom * 1e+26 if plot_denom > 0 else 0.0
                vals.append((th, p_ds, eu_ds, el_ds))
                t_vals.append((th, t_ds))
            ds_mid, ds_up, ds_low = ([], [], [])
            for th, p_ds, eu_ds, el_ds in vals:
                ds_mid.append(p_ds)
                if not np.all(np.isfinite([p_ds, eu_ds, el_ds])):
                    ds_up.append(np.nan)
                    ds_low.append(np.nan)
                    continue
                ds_up.append(p_ds + eu_ds)
                ds_low.append(max(0, p_ds - el_ds))
            t_ds_mid = [tv[1] for tv in t_vals]
            ax1.plot(theta_grid, ds_mid, color=c, label=f'Model ({mat})', linewidth=2)
            ax1.fill_between(theta_grid, ds_low, ds_up, color=c, alpha=0.2, label=f'Systematic band ({mat})')
            ax1.plot(theta_grid, t_ds_mid, color=c, linestyle='--', linewidth=1.5, alpha=0.8, label=f'Model w/o error ({mat})')
            sub_my = df_my[df_my['material'] == mat]
            if not sub_my.empty:
                th_arr = sub_my['angle'].values
                ds_meas_arr = sub_my['ds_meas'].values
                ds_meas_err_up_arr = sub_my['ds_meas_err_up'].values
                ds_meas_err_down_arr = sub_my['ds_meas_err_down'].values
                pull_arr = sub_my['pull'].values
                yerr_arr = np.array([ds_meas_err_down_arr, ds_meas_err_up_arr])
                ax1.errorbar(th_arr, ds_meas_arr, yerr=yerr_arr, fmt='o', color=c, markeredgecolor='black', label=f'My Data ({mat})', zorder=5, capsize=3)
                ax2.errorbar(th_arr, pull_arr, fmt='s', color=c, label=f'Pull ({mat})', zorder=4, capsize=3)
        group_constants_by_id = globals().get('group_calculated_constants', {})
        for f in all_files:
            g_id = str(f.get('group_id', '')).strip()
            if g_id == group_id or f.get('type') != 'Measurement' or g_id.lower() in exclude_group_ids:
                continue
            group_constants = group_constants_by_id.get(g_id)
            if not group_constants:
                print(f'[WARNING] Excluded other-group point: constants unavailable for group {g_id}.')
                continue
            mat = f.get('material', 'Au')
            isAu = mat == 'Au'
            th = float(f.get('angle', 0.0))
            t_sec = float(f.get('time_sec', 1.0))
            raw_data = f.get('raw_data', [])
            if raw_data and len(raw_data) >= 1024:
                total_cnt = float(np.sum(raw_data[0:1024]))
            else:
                total_cnt = float(f.get('total_count', 0.0))
            s1 = float(f.get('slit1', 4.0))
            s2 = float(f.get('slit2', 6.0))
            L = float(f.get('distance', 70.0))
            try:
                net_cnt, _, _ = observation_with_dead_time(total_cnt, t_sec)
            except ValueError as e:
                print(f'[WARNING] Excluded other-group point at {th} deg: {e}')
                continue
            L_s1 = L - 11.7 if L - 11.7 > 0 else 1.0
            omega_1 = np.pi * (s1 / 2.0) ** 2 * np.cos(np.radians(30.0)) / L_s1 ** 2
            omega_2 = np.pi * (s2 / 2.0) ** 2 / 60.0 ** 2
            n_target = group_constants.get('n_Au' if isAu else 'n_Al', 0.0)
            group_B_0 = group_constants.get('B_0', 0.0)
            incident_rate = group_B_0 * (omega_1 / (4.0 * np.pi))
            denom = incident_rate * n_target * omega_2 * t_sec
            if not 15.0 <= th <= 180.0 or not np.isfinite(denom) or denom <= 0 or (not np.isfinite(net_cnt)):
                print(f'[WARNING] Excluded other-group point at {th} deg: invalid value or uncertainty.')
                continue
            ds_val_cm2 = net_cnt / denom
            ds_val = ds_val_cm2 * 1e+26
            c_other = 'orange' if isAu else 'cyan'
            ax1.plot(th, ds_val, marker='x', color=c_other, alpha=0.3, markersize=6)
        if metric_a > 0:
            ax1.axvline(metric_a, color='gray', linestyle='--', label=f'Max Valid Angle a ({metric_a:.1f}°)')
        ax1.set_yscale('log')
        ax1.set_ylim(top=10000000.0)
        ax1.set_xlim(15, 120)
        ax1.set_title('Differential Cross Section & Model Comparison')
        ax1.set_ylabel('dσ/dΩ (fm²/sr)')
        ax1.grid(True, which='both', linestyle=':', alpha=0.6)
        ax1.legend(loc='upper right', fontsize=8)
        ax2.axhline(0, color='black', linestyle='-')
        ax2.axhline(1, color='red', linestyle=':', alpha=0.7, label='|Pull| = 1')
        ax2.axhline(-1, color='red', linestyle=':', alpha=0.7)
        ax2.set_xlim(15, 120)
        ax2.set_ylim(-3, 3)
        ax2.set_xlabel('Scattering Angle θ (deg)')
        ax2.set_ylabel('Pull')
        ax2.grid(True, linestyle=':', alpha=0.6)
        ax2.legend(loc='upper right', fontsize=8)
        plt.suptitle(model_caption())
        plt.tight_layout()
        plt.show()
        print('\n--- Class Comparison (a vs b Plot) ---')
        fig_comp, ax_comp = plt.subplots(figsize=(8, 6))
        comparison_id = experiment_state['personal_id'] if active_model_kind in ('Personal', 'Draft') else group_id
        all_metrics_list = [{'group_id': comparison_id, 'metric_a': metric_a, 'score_b': score_b}]
        for m in all_models:
            g_id = m.get('group_id', '')
            if g_id != comparison_id and str(m.get('metric_a', '')) != '' and (str(m.get('score_b', '')) != ''):
                if g_id.lower() in exclude_group_ids:
                    continue
                try:
                    all_metrics_list.append({'group_id': g_id, 'metric_a': float(m.get('metric_a')), 'score_b': float(m.get('score_b'))})
                except:
                    pass
        df_metrics_comp = pd.DataFrame(all_metrics_list)
        if not df_metrics_comp.empty:
            for idx, row in df_metrics_comp.iterrows():
                is_me = row['group_id'] == comparison_id
                col = 'red' if is_me else 'blue'
                sz = 120 if is_me else 60
                mk = '*' if is_me else 'o'
                ax_comp.scatter(row['metric_a'], row['score_b'], color=col, s=sz, marker=mk, label=row['group_id'] if is_me else '')
                ax_comp.annotate(row['group_id'], (row['metric_a'], row['score_b']), xytext=(5, 5), textcoords='offset points')
        ax_comp.set_title('Class Comparison: Max Valid Angle (a) vs Log-Likelihood Ratio (b)')
        ax_comp.set_xlabel('Max Valid Angle a (deg)')
        ax_comp.set_ylabel('Log-Likelihood Ratio b')
        ax_comp.grid(True, linestyle=':', alpha=0.6)
        if len(df_metrics_comp) > 0:
            ax_comp.legend()
        plt.suptitle(model_caption())
        plt.tight_layout()
        plt.show()
        print('--- Model Evaluation Completed Successfully ---')
        return evaluated_metrics
def run_personal_evaluation():
    global predict_counts_with_error
    try:
        require_link()
        predict_counts_with_error = compile_model(active_model_code)
        result = run_model_evaluation(upload=True)
        if result is not None:
            if 'render_trajectory' in globals():
                render_trajectory()
            if 'run_estimation' in globals():
                run_estimation(None)
        return result
    except Exception as exc:
        with out_model:
            print(f'[ERROR] {exc}')
        return None

display(out_model)
if linked:
    run_personal_evaluation()


In [ ]:
# @title 8.3 元データのプロット
sld_rebin = widgets.IntSlider(value=1, min=1, max=64, step=1, description='Rebin Factor:', continuous_update=False)
btn_plot_raw = widgets.Button(description='Plot Raw Spectra', button_style='primary', icon='line-chart')
out_raw = widgets.Output()

def plot_raw_spectra(*args):
    if not linked:
        return
    require_link()
    with out_raw:
        clear_output(wait=True)
        if 'experiment_state' not in globals() or 'uploaded_files' not in experiment_state:
            print('[ERROR] experiment_state is not initialized or no files uploaded.')
            return
        meas_files = [f for f in experiment_state['uploaded_files'].values() if f.get('type') == 'Measurement' and 'raw_data' in f and (len(f['raw_data']) > 0)]
        if not meas_files:
            print('[WARNING] No raw spectrum data found. Check the group measurement data.')
            return
        au_files = sorted([f for f in meas_files if f.get('material') == 'Au'], key=lambda x: float(x.get('angle', 0)))
        al_files = sorted([f for f in meas_files if f.get('material') == 'Al'], key=lambda x: float(x.get('angle', 0)))
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
        rebin_factor = sld_rebin.value

        def draw_spectra(ax, files, title):
            if not files:
                ax.text(0.5, 0.5, 'No data available', ha='center', va='center', transform=ax.transAxes, color='gray')
                ax.set_title(title)
                return
            angles = [float(f.get('angle', 0)) for f in files]
            min_ang, max_ang = (min(angles), max(angles))
            for f in files:
                ang = float(f.get('angle', 0))
                raw = f.get('raw_data', [])
                if len(raw) > 1024:
                    raw = raw[:1024]
                raw_arr = np.array(raw)
                if rebin_factor > 1:
                    new_len = len(raw_arr) // rebin_factor
                    trimmed_raw = raw_arr[:new_len * rebin_factor]
                    y_plot = np.sum(trimmed_raw.reshape(-1, rebin_factor), axis=1)
                    x_plot = np.arange(0, new_len * rebin_factor, rebin_factor) + (rebin_factor - 1) / 2.0
                else:
                    y_plot = raw_arr
                    x_plot = np.arange(len(raw_arr))
                if max_ang == min_ang:
                    color = 'C0'
                else:
                    color = cm.viridis((ang - min_ang) / (max_ang - min_ang))
                ax.step(x_plot, y_plot, where='mid', label=f'{ang:.0f}°', color=color, alpha=0.8, linewidth=1.5)
            ax.set_title(f'{title} (Rebin: x{rebin_factor})')
            ax.set_xlabel('Channel (ch)')
            ax.set_ylabel(f'Counts per {rebin_factor} ch')
            ax.set_yscale('log')
            ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Scattering Angle')
        draw_spectra(ax1, au_files, 'Au Target - Raw Spectra')
        draw_spectra(ax2, al_files, 'Al Target - Raw Spectra')
        fig.suptitle('Measurement group: ' + experiment_state['group_id'])
        plt.tight_layout()
        plt.show()
btn_plot_raw.on_click(plot_raw_spectra)
sld_rebin.observe(plot_raw_spectra, names='value')
display(widgets.VBox([widgets.HBox([btn_plot_raw, sld_rebin]), out_raw]))
if linked:
    plot_raw_spectra()


In [ ]:
# @title 9.1 イベント数の推定
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
out_estimate = widgets.Output()
lbl_est = widgets.HTML('<b>9.1 Input Device Settings for Prediction:</b><br>Estimate the expected scattering events and background events based on your updated model (Block 8.1).')
inp_theta_est = widgets.FloatText(value=20.0, description='Theta (deg):')
inp_time = widgets.FloatText(value=600.0, description='Time (sec):')
inp_s1 = widgets.Dropdown(options=[2.0, 4.0, 6.0], value=6.0, description='s1 (mm):')
inp_s2 = widgets.Dropdown(options=[2.0, 4.0, 6.0], value=6.0, description='s2 (mm):')
inp_L = widgets.Dropdown(options=[30.0, 50.0, 70.0], value=70.0, description='L (mm):')
dd_material = widgets.Dropdown(options=[('Gold (Au)', True), ('Aluminum (Al)', False)], description='Target:')
btn_estimate = widgets.Button(description='Predict Counts', button_style='primary')

def run_estimation(b):
    if not linked:
        return
    with out_estimate:
        clear_output()
        print(model_caption())
        print('--- Calculation Results ---')
        if 'experiment_state' not in globals():
            print('[ERROR] experiment_state is missing. Please run Block 0.1 / Block 6.0 first.')
            return
        user_predict_func = globals().get('predict_counts_with_error')
        if not callable(user_predict_func):
            print("[ERROR] 'predict_counts_with_error' is not defined. Please run Block 8.1 first.")
            return
        background_func = globals().get('background_count_estimate')
        if not callable(background_func):
            print('[ERROR] Please run Block 8.2 to define background_count_estimate first.')
            return
        theta_deg = inp_theta_est.value
        time_sec = inp_time.value
        s1 = inp_s1.value
        s2 = inp_s2.value
        L = inp_L.value
        isAu = dd_material.value
        if not np.isfinite(time_sec) or time_sec <= 0:
            print('[ERROR] Measurement time must be finite and greater than zero.')
            return
        try:
            p_cnt, stat_cnt, sys_err_plus, sys_err_minus = user_predict_func(theta_deg, time_sec, s1, s2, L, isAu)
            bg_expected, bg_stat_err = background_func(time_sec)
            total_expected = p_cnt + bg_expected
            mat_str = 'Au' if isAu else 'Al'
            print(f'Settings: Theta = {theta_deg} deg | Target = {mat_str} | Time = {time_sec} s')
            print(f'-------------------------------------------------------')
            print(f' Predicted Scattering Events : {p_cnt:.2f} ± {stat_cnt:.2f} (stat) +{sys_err_plus:.2f}/-{sys_err_minus:.2f} (syst) counts')
            print(f' Expected Background Events  : {bg_expected:.2f} ± {bg_stat_err:.2f} (stat) counts')
            print(f'-------------------------------------------------------')
            print(f' => Total Expected Observables: {total_expected:.2f} counts')
            if p_cnt > 0 and bg_expected / p_cnt > 0.5:
                print('\n[WARNING] Background is significant compared to the signal. Consider adjusting time or angle.')
        except Exception as e:
            print(f'ERROR calculating prediction: {e}')
btn_estimate.on_click(run_estimation)
ui_estimate = widgets.VBox([widgets.HBox([inp_theta_est, inp_time]), widgets.HBox([inp_s1, inp_s2, inp_L]), dd_material, btn_estimate, out_estimate])
display(ui_estimate)
if linked:
    measurements = [f for f in experiment_state['uploaded_files'].values() if f.get('type') == 'Measurement' and 15 <= float(f.get('angle', 0)) <= 180]
    if measurements:
        first = measurements[0]
        inp_theta_est.value = float(first['angle'])
        inp_time.value = float(first['time_sec'])
        inp_s1.value = float(first['slit1'])
        inp_s2.value = float(first['slit2'])
        inp_L.value = float(first['distance'])
        dd_material.value = first['material'] == 'Au'
    run_estimation(None)
